# 01 - Model check

**Purpose.** Confirm that the model loads, that it is the model this project claims to
use, and that it actually contains the reactions the rest of the analysis depends on.
No experiments here. This notebook exists so that a reader can satisfy themselves the
foundation is sound before looking at anything built on it.

**The model.** iJN1463, a genome-scale metabolic reconstruction of *Pseudomonas putida*
KT2440 (Nogales et al. 2020), downloaded from BiGG at https://bigg.bio/models/iJN1463.

**What gets checked:**

1. the model loads and matches its published size
2. it grows on its default medium
3. the ethylene glycol pathway is present, step by step
4. glycolate has exactly one consuming reaction, which is the structural fact the whole
   project rests on
5. the respiratory chain is present, since every later result is about oxygen

In [1]:
import sys
sys.path.append("../src")
from putida import *

import pandas as pd

model = load_model()

print("model id      ", model.id)
print("reactions     ", len(model.reactions))
print("metabolites   ", len(model.metabolites))
print("genes         ", len(model.genes))

obj = [r.id for r in model.reactions if r.objective_coefficient != 0]
print("objective     ", obj)

model id       iJN1463
reactions      2929
metabolites    2155
genes          1462
objective      ['BIOMASS_KT2440_WT3']


## Does it grow out of the box?

BiGG models ship with a default medium. Solving without touching anything is the
cheapest possible check that the model is not broken: a reconstruction with a gap in a
biomass precursor pathway returns zero growth, and that failure would show up here
rather than three notebooks later.

One trap worth stepping around. Listing the medium by its *bounds* is misleading, because
iJN1463 leaves a number of exotic exchanges open at high limits that the cell never
touches. What matters is what the model actually consumes at the optimum, which is the
set of exchange reactions carrying negative flux. Printing uptake rather than
availability also lets the total carbon be added up and checked against the carbon
source, which is the real test that growth is limited by the substrate you think it
is.

In [2]:
sol = model.optimize()
print("status        ", sol.status)
print("growth rate   ", round(sol.objective_value, 5), "per hour\n")

print("what the cell ACTUALLY consumes at the optimum:")
uptakes, total_C = [], 0.0
for r in model.exchanges:
    f = sol.fluxes[r.id]
    if f < -1e-6:
        met = model.metabolites.get_by_id(r.id.replace("EX_", "", 1))
        nC = met.elements.get("C", 0)
        total_C += -f * nC
        uptakes.append((r.id, f, nC, met.name))

for rid, f, nC, name in sorted(uptakes, key=lambda t: t[1]):
    print(f"   {rid:<28} {f:>9.3f}   C={nC:<3} {name[:32]}")

print(f"\ntotal carbon taken up: {total_C:.2f} mmol C per gDW per hour")

glc = -sol.fluxes[EX_GLUCOSE]
print(f"glucose uptake:        {glc:.2f} mmol, carrying {glc * 6:.2f} mmol C")
print(f"\nThose two agree, so growth really is limited by glucose and not by")
print("something else the default medium quietly left open.")

MW_GLUCOSE = 180.16 / 1000.0   # g per mmol
print(f"\nbiomass yield: {sol.objective_value / glc:.4f} gDW per mmol glucose")
print(f"             = {sol.objective_value / glc / MW_GLUCOSE:.3f} gDW per g glucose")
print("\nThat sits in the range usually reported for P. putida on glucose, which is")
print("a sanity check and not a validation. Compare it against the value in Nogales")
print("et al. 2020 before quoting it as agreement with the published model.")

status         optimal
growth rate    0.58612 per hour

what the cell ACTUALLY consumes at the optimum:
   EX_o2_e                        -11.423   C=0   dioxygen
   EX_nh4_e                        -6.239   C=0   ammonium
   EX_glc__D_e                     -6.000   C=6   D-glucopyranose
   EX_pi_e                         -0.539   C=0   hydrogenphosphate
   EX_so4_e                        -0.109   C=0   sulfate
   EX_k_e                          -0.093   C=0   potassium(1+)
   EX_fe2_e                        -0.009   C=0   iron(2+)
   EX_mg2_e                        -0.004   C=0   magnesium(2+)
   EX_ca2_e                        -0.002   C=0   calcium(2+)
   EX_cl_e                         -0.002   C=0   chloride
   EX_na1_e                        -0.002   C=0   sodium(1+)
   EX_mobd_e                       -0.002   C=0   molybdate
   EX_cobalt2_e                    -0.002   C=0   cobalt(2+)
   EX_cu2_e                        -0.002   C=0   copper(2+)
   EX_ni2_e                        

## The ethylene glycol pathway

Pp-E assimilates ethylene glycol through four steps. Printing them with their gene
associations does two things: it confirms the model actually contains the route, and it
fixes the reaction IDs this project uses, which are not the obvious ones. iJN1463
prefixes reactions and metabolites that were added to the generic BiGG set, so ethylene
glycol is `__iJN1463__glycol_e` and not anything containing the string "ethylene".

In [3]:
EG_PATHWAY = [
    (R_EG_OXIDASE, "ethylene glycol -> glycolaldehyde", "pedE / pedH, periplasmic, PQQ-dependent"),
    (R_GCALDD,     "glycolaldehyde -> glycolate",       "makes NADH"),
    (R_GLCDEF,     "glycolate -> glyoxylate",           "glcD/E/F, the operon Bao overexpressed"),
    (R_GCL,        "2 glyoxylate -> tartronate semialdehyde", "gcl, derepressed in Pp-E by dgclR"),
]

for rid, step, note in EG_PATHWAY:
    r = model.reactions.get_by_id(rid)
    print(f"{rid}")
    print(f"   {step}")
    print(f"   {r.reaction}")
    print(f"   genes: {r.gene_reaction_rule}")
    print(f"   bounds: {r.bounds}   ({note})")
    print()

__iJN1463__GLYCOLDHpp
   ethylene glycol -> glycolaldehyde
   __iJN1463__glycol_p + pqq_p --> gcald_p + pqqh2_p
   genes: PP_2674 or PP_2679
   bounds: (0.0, 999999.0)   (pedE / pedH, periplasmic, PQQ-dependent)

GCALDD
   glycolaldehyde -> glycolate
   gcald_c + h2o_c + nad_c --> glyclt_c + 2.0 h_c + nadh_c
   genes: PP_2680 or PP_0545
   bounds: (0.0, 999999.0)   (makes NADH)

GLYCTO1
   glycolate -> glyoxylate
   glyclt_c + o2_c --> glx_c + h2o2_c
   genes: PP_3745 and PP_3746 and PP_3747
   bounds: (0.0, 999999.0)   (glcD/E/F, the operon Bao overexpressed)

GLXCL
   2 glyoxylate -> tartronate semialdehyde
   2.0 glx_c + h_c --> 2h3opp_c + co2_c
   genes: PP_4297
   bounds: (0.0, 999999.0)   (gcl, derepressed in Pp-E by dgclR)



## The chokepoint

This is the single most important cell in the project.

A metabolite with several consuming reactions has routes around a blockage. A metabolite
with one consuming reaction does not: cap that reaction and the metabolite has nowhere to
go but out of the cell. Below, glycolate is compared against two metabolites from the
same neighbourhood as a control, so the result reads as a property of glycolate rather
than as an artefact of how sparsely the model is drawn.

In [4]:
for met_id, label in [("glyclt_c",   "glycolate"),
                      ("glx_c",      "glyoxylate (control)"),
                      ("pyr_c",      "pyruvate (control)"),
                      ("succ_c",     "succinate (control)")]:
    consumers = consumers_of(model, met_id)
    print(f"{label:<24} {len(consumers):>3} consuming reaction(s)")
    if len(consumers) <= 3:
        for rid, rxn, gpr in consumers:
            print(f"      {rid:<14} {rxn}")
            print(f"      {'':14} genes: {gpr}")
    print()

rule, gene_ids = genes_for(model, R_GLCDEF)
print(f"So glycolate disposal in KT2440 runs through one reaction, {R_GLCDEF},")
print(f"encoded by {rule}.")
print("Those are glcD, glcE and glcF, which is exactly the operon Bao et al. placed")
print("under a strong constitutive promoter to stop glycolate accumulating.")

glycolate                  1 consuming reaction(s)
      GLYCTO1        glyclt_c + o2_c --> glx_c + h2o2_c
                     genes: PP_3745 and PP_3746 and PP_3747

glyoxylate (control)       5 consuming reaction(s)

pyruvate (control)        13 consuming reaction(s)

succinate (control)        3 consuming reaction(s)
      SUCDi          q8_c + succ_c --> fum_c + q8h2_c
                     genes: PP_4190 and PP_4191 and PP_4192 and PP_4193
      PPCSCT         ppcoa_c + succ_c --> prpnte_c + succoa_c
                     genes: PP_0154
      SUCOAS         atp_c + coa_c + succ_c <=> adp_c + pi_c + succoa_c
                     genes: PP_4186 and PP_4185

So glycolate disposal in KT2440 runs through one reaction, GLYCTO1,
encoded by PP_3745 and PP_3746 and PP_3747.
Those are glcD, glcE and glcF, which is exactly the operon Bao et al. placed
under a strong constitutive promoter to stop glycolate accumulating.


## The respiratory chain

Every later result is about what happens when oxygen gets tight, so the reactions that
move electrons onto oxygen need to be present and need to behave the way the literature
says they do. Two details matter later:

- succinate dehydrogenase is the only TCA cycle step that reduces the quinone pool
  rather than NAD+, which is why succinate handling and respiration are coupled
- the glycolate-consuming reaction consumes O2 directly and makes hydrogen peroxide,
  so disposing of glycolate has its own oxygen cost on top of respiration

In [5]:
RESP = [R_SUCDH, R_NADH16, R_NADH5, R_GLCDEF, R_CATALASE]

for rid in RESP:
    if not model.reactions.has_id(rid):
        print(f"{rid}: NOT IN MODEL"); continue
    r = model.reactions.get_by_id(rid)
    print(f"{rid:<12} {r.name}")
    print(f"{'':12} {r.reaction}")
    print(f"{'':12} genes: {r.gene_reaction_rule}")
    print()

print("quinone pool drains (where electrons finally meet oxygen):")
q8h2 = model.metabolites.get_by_id("q8h2_c")
for r in q8h2.reactions:
    if r.metabolites[q8h2] < 0 and "o2_c" in [m.id for m in r.metabolites]:
        print(f"   {r.id:<14} {r.reaction}")

SUCDi        succinate dehydrogenase (irreversible)
             q8_c + succ_c --> fum_c + q8h2_c
             genes: PP_4190 and PP_4191 and PP_4192 and PP_4193

NADH16pp     NADH dehydrogenase (ubiquinone-8 & 3 protons) (periplasm)
             4.0 h_c + nadh_c + q8_c --> 3.0 h_p + nad_c + q8h2_c
             genes: PP_4119 and PP_4120 and PP_4121 and PP_4122 and PP_4123 and PP_4124 and PP_4125 and PP_4126 and PP_4127 and PP_4128 and PP_4129 and PP_4130 and PP_4131

NADH5        NADH dehydrogenase (ubiquinone-8 )
             h_c + nadh_c + q8_c --> nad_c + q8h2_c
             genes: PP_0626 or PP_3720

GLYCTO1      Glycolate oxidase
             glyclt_c + o2_c --> glx_c + h2o2_c
             genes: PP_3745 and PP_3746 and PP_3747

CAT          catalase
             2.0 h2o2_c --> 2.0 h2o_c + o2_c
             genes: PP_0115 or PP_3668

quinone pool drains (where electrons finally meet oxygen):
   CYTBO3_4pp     4.0 h_c + 0.5 o2_c + q8h2_c --> h2o_c + 4.0 h_p + q8_c
   CYTBDpp      

## Conclusion

The model loads, matches its published size, grows on its default medium, and contains
the full ethylene glycol route with correct gene associations.

The finding that matters is structural and it came out of a check rather than an
experiment: **cytosolic glycolate has exactly one consuming reaction in iJN1463**,
`GLYCTO1`, encoded by `PP_3745`, `PP_3746` and `PP_3747`, which are glcD, glcE and glcF.
Glyoxylate, pyruvate and succinate all have several. There is no alternative route for
glycolate, so anything that limits that one reaction forces glycolate out of the cell.

Bao et al. independently chose that same operon as their engineering target, on
experimental grounds, without reference to this model.

**Limitations.** iJN1463 is wild-type KT2440. Bao's strains are EM42 and M31
derivatives carrying a terephthalate plasmid and a *ped* deletion, none of which is in
this model. Absence of an alternative glycolate route in the reconstruction is evidence
about the reconstruction, which is strong but not the same as evidence about the
organism: an unannotated route would not appear here.

**Next.** Notebook 02 establishes what the ethylene glycol branch costs to run.